## Import Libraries & Load Data

In [21]:
import pandas as pd
import numpy as np

file_path = 'data_resources/retail_store_sales.csv'

df = pd.read_csv(file_path)
print("✅ Data loaded successfully!")
print(f"Total records: {df.shape[0]} rows, {df.shape[1]} columns")

✅ Data loaded successfully!
Total records: 12575 rows, 11 columns


## Data Exploration

In [22]:
display(df.head())

print("\n--- Missing Values Summary ---")
print(df.isnull().sum())

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False



--- Missing Values Summary ---
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


## Case 3 - Price Manipulation Detection

In [23]:
df_case3 = df.copy()

mask_not_null = df_case3[['Price Per Unit', 'Quantity', 'Total Spent']].notnull().all(axis=1)

mask_price_manipulated = round(df_case3['Price Per Unit'] * df_case3['Quantity'], 2) != round(df_case3['Total Spent'], 2)

anomalies_case3 = df_case3[mask_not_null & mask_price_manipulated]

print(f"🚨 Found suspicious price manipulations: {len(anomalies_case3)} transactions")
display(anomalies_case3[['Transaction ID', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent']].head())

🚨 Found suspicious price manipulations: 0 transactions


,Transaction ID,Item,Price Per Unit,Quantity,Total Spent


## Case 2 - Ghost Transactions Detection

In [24]:
df_case2 = df.copy()

mask_ghost_txn = df_case2['Price Per Unit'].isna() | df_case2['Quantity'].isna() | df_case2['Total Spent'].isna()

anomalies_case2 = df_case2[mask_ghost_txn]

print(f"🚨 Found ghost transactions (missing critical data): {len(anomalies_case2)} transactions")
display(anomalies_case2.head())

🚨 Found ghost transactions (missing critical data): 1213 transactions


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
5,TXN_7482416,CUST_09,Patisserie,NaN,NaN,10.0,200.0,Credit Card,Online,2023-11-30,NaN
7,TXN_1372952,CUST_21,Furniture,NaN,33.5,NaN,NaN,Digital Wallet,In-store,2024-04-02,True
11,TXN_5422631,CUST_09,Milk Products,NaN,NaN,8.0,52.0,Digital Wallet,In-store,2025-01-12,True
15,TXN_1809665,CUST_14,Beverages,NaN,24.5,NaN,NaN,Credit Card,In-store,2022-05-11,NaN
17,TXN_9634894,CUST_15,Milk Products,NaN,NaN,10.0,275.0,Digital Wallet,Online,2022-04-17,NaN


## Case 1 - Data Synthesizing (Mocking Time Data)

In [25]:
df_case1 = df.copy()

num_rows = len(df_case1)
num_normal = int(num_rows * 0.95)
num_anomaly = num_rows - num_normal

def random_time(start_hour, end_hour, size):
    hours = np.random.randint(start_hour, end_hour, size)
    minutes = np.random.randint(0, 60, size)
    seconds = np.random.randint(0, 60, size)
    return [f"{h:02d}:{m:02d}:{s:02d}" for h, m, s in zip(hours, minutes, seconds)]

normal_times = random_time(8, 22, num_normal)
anomaly_times = random_time(1, 4, num_anomaly)

all_times = normal_times + anomaly_times
np.random.shuffle(all_times)

df_case1['Transaction Time'] = all_times

print("✅ Successfully synthesized 'Transaction Time' column")
display(df_case1[['Transaction ID', 'Transaction Date', 'Transaction Time']].head())

✅ Successfully synthesized 'Transaction Time' column


,Transaction ID,Transaction Date,Transaction Time
0,TXN_6867343,2024-04-08,18:28:21
1,TXN_3731986,2023-07-23,14:58:57
2,TXN_9303719,2022-10-05,20:37:27
3,TXN_9458126,2022-05-07,10:52:36
4,TXN_4575373,2022-10-02,18:08:27


## NLP Case Matching (TF-IDF & Cosine Similarity)

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

cases = {
    "Case 1": "Midnight sales, transaction out of office hours, late night shop closed, time anomaly.",
    "Case 2": "Ghost transactions, missing data, voided bills, null values, deleted items, cash fraud.",
    "Case 3": "Price manipulation, mismatch total spent, discount abuse, changed unit price, calculation error."
}

case_names = list(cases.keys())
case_descriptions = list(cases.values())

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(case_descriptions)

def match_case(user_query):
    """
    Function to match a user's text query to the most relevant fraud case.
    """
    query_vec = vectorizer.transform([user_query])

    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    best_match_idx = np.argmax(similarities)
    best_score = similarities[best_match_idx]

    if best_score < 0.1:
        return "Unknown", best_score

    return case_names[best_match_idx], best_score

test_queries = [
    "I suspect my cashier is deleting bills to take cash.",
    "Why is there a sale at 3 AM when the store is closed?",
    "The total amount doesn't match the item price."
]

print("--- TF-IDF Case Matching Results ---")
for query in test_queries:
    matched_case, score = match_case(query)
    print(f"User Query : '{query}'")
    print(f"Matched    : {matched_case} (Confidence Score: {score:.4f})\n")

--- TF-IDF Case Matching Results ---
User Query : 'I suspect my cashier is deleting bills to take cash.'
Matched    : Case 2 (Confidence Score: 0.4082)

User Query : 'Why is there a sale at 3 AM when the store is closed?'
Matched    : Case 1 (Confidence Score: 0.3015)

User Query : 'The total amount doesn't match the item price.'
Matched    : Case 3 (Confidence Score: 0.5669)



In [27]:
df_final = df_case1.copy()

np.random.seed(42)
valid_indices = df_final[['Price Per Unit', 'Quantity', 'Total Spent']].dropna().index
manipulate_indices = np.random.choice(valid_indices, size=50, replace=False)

df_final.loc[manipulate_indices, 'Total Spent'] = df_final.loc[manipulate_indices, 'Total Spent'] * 0.5

df_final['Discount Applied'] = df_final['Discount Applied'].fillna(False)

df_final['Timestamp'] = pd.to_datetime(df_final['Transaction Date'] + ' ' + df_final['Transaction Time'])

print(f"✅ Feature Engineering completed. Dataset shape: {df_final.shape}")
display(df_final.head())

df_final.to_csv('data_resources/retail_store_sales_engineered.csv', index=False)
print("💾 Successfully saved retail_store_sales_engineered.csv!")

✅ Feature Engineering completed. Dataset shape: (12575, 13)


C:\Users\Nitro V15\AppData\Local\Temp\ipykernel_26092\818517136.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final['Discount Applied'] = df_final['Discount Applied'].fillna(False)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Transaction Time,Timestamp
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,18:28:21,2024-04-08 18:28:21
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,14:58:57,2023-07-23 14:58:57
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,20:37:27,2022-10-05 20:37:27
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,False,10:52:36,2022-05-07 10:52:36
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,18:08:27,2022-10-02 18:08:27


💾 Successfully saved retail_store_sales_engineered.csv!


## Advanced Thai NLP Matching (PyThaiNLP + TF-IDF)

In [28]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pythainlp.tokenize import word_tokenize

def thai_tokenizer(text):
    return word_tokenize(text, engine='newmm', keep_whitespace=False)

cases_dict = {
    "Case 1": "ขายของตอนดึก แอบขายตอนร้านปิด มีบิลโผล่มาตอนเที่ยงคืน ยอดเข้าตอนกลางคืน นอกเวลาทำงาน แอบเปิดเครื่องคิดเงิน",
    "Case 2": "ยกเลิกรายการ ลบของออกจากบิล แอบลบรายการแล้วรับเงินสด ลบบิลทิ้ง เงินในเก๊ะไม่ครบ ยกเลิกออเดอร์ แอบขโมยเงิน",
    "Case 3": "ยอดเงินไม่ตรงกับของที่ขาย แอบลดราคาให้เพื่อน ขายของถูกกว่าป้าย คำนวณเงินผิด เงินได้น้อยกว่าของที่ออกไป ยอดรวมไม่ตรง"
}

case_names = list(cases_dict.keys())
case_descriptions = list(cases_dict.values())

vectorizer = TfidfVectorizer(tokenizer=thai_tokenizer)
tfidf_matrix = vectorizer.fit_transform(case_descriptions)

def match_fraud_case_thai(user_query):
    """
    Match a natural Thai user query to the predefined fraud cases.
    """
    query_vec = vectorizer.transform([user_query])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    best_idx = np.argmax(similarities)
    best_score = similarities[best_idx]

    if best_score < 0.05: # Threshold for no match
        return "Unknown Case", best_score
    return case_names[best_idx], best_score

test_queries = [
    "สงสัยว่าลูกน้องจะแอบลบรายการทิ้งแล้วเอาเงินเข้ากระเป๋าตัวเอง",
    "ร้านปิดสี่ทุ่ม แต่ทำไมมีบิลเด้งมาตอนตีสองตีสาม",
    "รวมเงินแล้วยอดไม่ตรงกับของที่ขายไปเลย เหมือนขายถูกลง",
    "มีบิลเด้งขึ้นมาหลังจากร้านปิดแล้ว แอบเปิดเครื่องคิดเงินตอนดึก",
    "วันนี้อากาศร้อนมาก"
]

print("--- Advanced Thai NLP Matching Results ---")
for query in test_queries:
    matched_case, score = match_fraud_case_thai(query)
    print(f"Query   : '{query}'")
    print(f"Matched : {matched_case} (Confidence Score: {score:.4f})\n")

--- Advanced Thai NLP Matching Results ---
Query   : 'สงสัยว่าลูกน้องจะแอบลบรายการทิ้งแล้วเอาเงินเข้ากระเป๋าตัวเอง'
Matched : Case 2 (Confidence Score: 0.6221)

Query   : 'ร้านปิดสี่ทุ่ม แต่ทำไมมีบิลเด้งมาตอนตีสองตีสาม'
Matched : Case 1 (Confidence Score: 0.5921)

Query   : 'รวมเงินแล้วยอดไม่ตรงกับของที่ขายไปเลย เหมือนขายถูกลง'
Matched : Case 3 (Confidence Score: 0.6098)

Query   : 'มีบิลเด้งขึ้นมาหลังจากร้านปิดแล้ว แอบเปิดเครื่องคิดเงินตอนดึก'
Matched : Case 1 (Confidence Score: 0.6258)

Query   : 'วันนี้อากาศร้อนมาก'
Matched : Unknown Case (Confidence Score: 0.0000)



C:\Users\Nitro V15\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
